In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from scipy.interpolate import griddata
import glob
import time

# 设置绘图样式
plt.rcParams.update({
    'font.size': 22,  
    'font.weight': 'bold', 
    'axes.titlesize': 22, 
    'axes.labelsize': 22,
    'axes.titleweight': 'bold', 
    'axes.labelweight': 'bold',  
    'legend.fontsize': 22,

    # 添加全局线宽设置
    'axes.linewidth': 2,         # 坐标轴线宽
    'grid.linewidth': 1.5,      # 网格线宽
    'lines.linewidth': 2.5,     # 曲线线宽
    'xtick.major.width': 2,     # X轴主刻度线宽
    'ytick.major.width': 2,     # Y轴主刻度线宽
    'hatch.linewidth': 2,       # 填充图案线宽
})

# ================= 加载训练结果 =================
RESULTS_ROOT = "./pinn_results"

# 查找最新的PINN运行目录
pinn_runs = sorted(glob.glob(os.path.join(RESULTS_ROOT, "run_pi100_K50_*")))
if not pinn_runs:
    raise FileNotFoundError("No PINN results directory found")
pinn_dir = os.path.join(RESULTS_ROOT, "run_pi100_K50_20250809-230245")  # 使用最新的运行
RESULT_DIR=pinn_dir
SAVE_DIR = "./generated/one-period-result-fig"
print(f"Loading results from: {pinn_dir}")

# 加载训练损失
loss_path = os.path.join(RESULT_DIR, "training_loss.npy")
train_loss = np.load(loss_path)

# 加载预测结果
results_path = os.path.join(RESULT_DIR, "pinn_results.npy")
results = np.load(results_path)
A_vals = results[:, 0]
t_vals = results[:, 1]
C_vals = results[:, 2]

# 创建插值网格
A_min, A_max = np.min(A_vals), np.max(A_vals)
t_min, t_max = np.min(t_vals), np.max(t_vals)

A_plot = np.linspace(A_min, A_max, 100)
t_plot = np.linspace(t_min, t_max, 100)
A_mesh, t_mesh = np.meshgrid(A_plot, t_plot, indexing='ij')

# 插值选项价格
C_grid = griddata((A_vals, t_vals), C_vals, (A_mesh, t_mesh), method='cubic')

# ================= 重新创建模型用于分析 =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class CarbonOptionNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.Tanh(),
            nn.Linear(16, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1),
        )

    def forward(self, A, t):
        if A.dim() == 1:
            A = A.view(-1, 1)
        if t.dim() == 1:
            t = t.view(-1, 1)
        x = torch.cat([A, t], dim=1)
        return self.net(x)

# 加载模型
model = CarbonOptionNN().to(device)
model_path = os.path.join(RESULT_DIR, "carbon_option_model.pth")
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("Model loaded successfully")

# ================= 绘图1：综合结果图 =================
print("Creating comprehensive results plot...")
start_time = time.time()

# 时间点选择
time_points = np.arange(0, t_max + 0.1, 0.5)
colors = cm.viridis(np.linspace(0, 1, len(time_points)))

# 创建图窗
# fig = plt.figure(figsize=(32, 8))
# 创建图窗并调整整体布局参数
fig = plt.figure(figsize=(34, 8))  # 宽度增加让3D图有更大空间
plt.subplots_adjust(top=0.88)      # 为标题预留更多顶部空间

# 使用GridSpec进行更精确的子图布局控制
gs = gridspec.GridSpec(1, 4, width_ratios=[1.2, 1.0, 1.0, 1.0])  # 增大第一个图宽度比例
# 子图1：3D表面图
ax1 = fig.add_subplot(gs[0], projection='3d')
surf = ax1.plot_surface(A_mesh, t_mesh, C_grid, cmap='viridis', alpha=0.9)
ax1.set_xlabel('Asset Price (A)',labelpad=18)
ax1.set_ylabel('Time (t)',labelpad=18)
ax1.set_zlabel('Option Price (C)',labelpad=18)
ax1.set_title('PINN Solution for Carbon Option', pad=20)
# fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10, label='Option Price')

ax2 = fig.add_subplot(gs[1])
contour = ax2.contourf(A_mesh, t_mesh, C_grid, levels=50, cmap="viridis")
ax2.contour(A_mesh,  t_mesh, C_grid,  levels=10, colors='k', linewidths=0.5, alpha=0.5)
cbar = plt.colorbar(contour, ax=ax2)
cbar.set_label("Option Price (C)")
ax2.set_xlabel("Asset Price (A)")
ax2.set_ylabel("Time (t)")
ax2.set_title("2D Potential Surface of Option Price", pad=20)
ax2.grid(True, linestyle="--", alpha=0.5)

# 子图3：2D势能面图
ax3 = fig.add_subplot(gs[2])
contour = ax3.contourf(A_mesh, C_grid, t_mesh, levels=50, cmap="viridis")
ax3.contour(A_mesh, C_grid, t_mesh, levels=10, colors='k', linewidths=0.5, alpha=0.5)
cbar = plt.colorbar(contour, ax=ax3)
cbar.set_label("Time (t)")
ax3.set_xlabel("Asset Price (A)")
ax3.set_ylabel("Option Price (C)")
ax3.set_title("2D Potential Surface of Option Price", pad=20)
ax3.grid(True, linestyle="--", alpha=0.5)

# 子图2：价格曲线图
ax4 = fig.add_subplot(gs[3])

# 创建规则网格用于绘制价格曲线
A_plot_curve = np.linspace(A_min, A_max, 100)
t_plot_curve = time_points

# 创建网格用于预测
A_flat_plot, t_flat_plot = np.meshgrid(A_plot_curve, t_plot_curve, indexing='ij')
A_flat_plot = A_flat_plot.ravel()
t_flat_plot = t_flat_plot.ravel()

# 转换为张量
A_tensor_plot = torch.tensor(A_flat_plot, dtype=torch.float32, device=device).view(-1, 1)
t_tensor_plot = torch.tensor(t_flat_plot, dtype=torch.float32, device=device).view(-1, 1)

# 预测
with torch.no_grad():
    C_pred_plot = model(A_tensor_plot, t_tensor_plot).cpu().numpy().reshape(len(A_plot_curve), len(t_plot_curve))

# 绘制每条曲线
for i, (t_val, color) in enumerate(zip(t_plot_curve, colors)):
    ax4.plot(A_plot_curve, C_pred_plot[:, i], 
             label=f"t = {t_val:.1f}", 
             linewidth=2, 
             color=color)

ax4.set_xlabel("Asset Price (A)")
ax4.set_ylabel("Option Price (C)")
ax4.set_title("Option Price vs Asset Price at Different Time Points", pad=20)
ax4.legend()
ax4.grid(True, alpha=0.3)



plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "PINN-result.png"), dpi=300)
print(f"Comprehensive plot saved in {time.time()-start_time:.2f} seconds")
plt.show()

# ================= 计算并绘制残差 =================
print("Calculating and plotting PDE residuals...")
start_time = time.time()

# 重新定义计算PDE残差的函数（修复梯度问题）
def pde_residual(model, A, t, pi=100.0, beta=0.5, alpha=1.2, r=0.05, T=4.0, tau=3.0):
    # 确保输入张量需要梯度
    A = A.clone().detach().requires_grad_(True)
    t = t.clone().detach().requires_grad_(True)
    
    # 计算导数
    C = model(A, t)
    
    # 计算dC_dt
    dC_dt = torch.autograd.grad(
        C, t, 
        grad_outputs=torch.ones_like(C),
        create_graph=True,
        retain_graph=True
    )[0]
    
    # 计算dC_dA
    dC_dA = torch.autograd.grad(
        C, A, 
        grad_outputs=torch.ones_like(C),
        create_graph=True,
        retain_graph=True
    )[0]
    
    # 计算d2C_dA2
    d2C_dA2 = torch.autograd.grad(
        dC_dA, A, 
        grad_outputs=torch.ones_like(dC_dA),
        create_graph=True,
        retain_graph=True
    )[0]
    
    # 计算相关项
    z_t = beta * (T - t).pow(-alpha)
    A_pi_ratio = A / pi
    A_pi_ratio = A_pi_ratio.clamp(min=1e-6, max=1-1e-6)
    
    # 使用标准正态分布的逆CDF
    Phi_inv = torch.distributions.normal.Normal(0, 1).icdf(A_pi_ratio)
    Phi_prime = torch.exp(-0.5 * Phi_inv.pow(2)) / torch.sqrt(torch.tensor(2.0 * torch.pi))
    
    # 非线性项
    nonlinear_term = 0.5 * d2C_dA2 * (pi * Phi_prime * torch.sqrt(z_t)).pow(2)
    
    # PDE残差
    residual = dC_dt + nonlinear_term - r * C
    
    return residual

# 计算残差
n_residual = 500  # 减少点数以提高速度
residual_A = np.linspace(A_min, A_max, n_residual)
residual_t = np.linspace(t_min, t_max, n_residual)

# 创建网格
residual_A_tensor = torch.tensor(residual_A, dtype=torch.float32, device=device).view(-1, 1)
residual_t_tensor = torch.tensor(residual_t, dtype=torch.float32, device=device).view(-1, 1)

# 计算残差
residual = pde_residual(model, residual_A_tensor, residual_t_tensor).cpu().detach().numpy()

# 转换为NumPy数组
residual_A_np = residual_A_tensor.cpu().numpy().flatten()
residual_t_np = residual_t_tensor.cpu().numpy().flatten()

# 可视化残差
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# A 与残差的关系
ax1.scatter(residual_A_np, residual, alpha=0.6, color="blue")
ax1.axhline(0, color="black", linestyle="--")
ax1.set_title("Residual vs Asset Price (A)")
ax1.set_xlabel("Asset Price (A)")
ax1.set_ylabel("Residual")
ax1.set_ylim(-10, 10)

ax1.grid(True)

# t 与残差的关系
ax2.scatter(residual_t_np, residual, alpha=0.6, color="red")
ax2.axhline(0, color="black", linestyle="--")
ax2.set_title("Residual vs Time (t)")
ax2.set_xlabel("Time (t)")
ax2.set_ylabel("Residual")
ax2.set_ylim(-5, 5)
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "pde_residuals.png"), dpi=300)
print(f"Residual plot saved in {time.time()-start_time:.2f} seconds")
plt.show()

# ================= 边界条件检查 =================
print("\nChecking boundary conditions...")
start_time = time.time()

# 从参数文件中提取参数
params_path = os.path.join(SAVE_DIR, "simulation_parameters.txt")
if os.path.exists(params_path):
    with open(params_path, "r") as f:
        params_info = f.read()
        # 提取参数值
        pi = float(params_info.split("Pi = ")[1].split("\n")[0])
        beta = float(params_info.split("Beta = ")[1].split("\n")[0])
        alpha = float(params_info.split("Alpha = ")[1].split("\n")[0])
        r = float(params_info.split("Risk-free rate (r) = ")[1].split("\n")[0])
        T = float(params_info.split("T = ")[1].split("\n")[0])
        tau = float(params_info.split("Tau = ")[1].split("\n")[0])
        K = float(params_info.split("Strike price (K) = ")[1].split("\n")[0])
else:
    # 默认参数
    pi = 100.0
    beta = 0.5
    alpha = 0.8
    r = 0.05
    T = 4.0
    tau = 3.0
    K = 50.0

print(f"Using parameters: pi={pi}, beta={beta}, alpha={alpha}, r={r}, T={T}, tau={tau}, K={K}")

# 创建测试点
t_test = torch.linspace(t_min, t_max, 10, device=device).view(-1, 1)
A_min_test = torch.zeros_like(t_test, device=device)
A_max_test = torch.ones_like(t_test, device=device) * A_max
A_mid = torch.ones_like(t_test, device=device) * 0

with torch.no_grad():
    # 边界条件 A = 0
    C_at_A_min = model(A_min_test, t_test)
    
    # 边界条件 A = 100
    C_at_A_max = model(A_max_test, t_test)
    expected_at_A_max = A_max_test - K * torch.exp(-r * (tau - t_test))
    
    # 终端条件 t = tau
    t_end = torch.full_like(A_mid, tau, device=device)
    final_pred = model(A_mid, t_end)
    final_expected = torch.maximum(A_mid - K, torch.tensor(0.0, device=device))
    
    # 初始条件 t = 0
    t_start = torch.full_like(A_mid, 0.0, device=device)
    C_at_t_start = model(A_mid, t_start)

# 准备边界条件数据
boundary_data = [
    {
        "Condition": "Boundary (A=0)",
        "Value": f"{C_at_A_min.mean().item():.6f}",
        "Expected": "≈0",
        "Difference": f"{C_at_A_min.abs().mean().item():.6f}"
    },
    {
        "Condition": "Boundary (A=100)",
        "Value": f"{C_at_A_max.mean().item():.6f}",
        "Expected": f"{expected_at_A_max.mean().item():.6f}",
        "Difference": f"{(C_at_A_max - expected_at_A_max).abs().mean().item():.6f}"
    },
    {
        "Condition": "Terminal (t=τ)",
        "Value": f"{final_pred.mean().item():.6f}",
        "Expected": f"{final_expected.mean().item():.6f}",
        "Difference": f"{(final_pred - final_expected).abs().mean().item():.6f}"
    },
    {
        "Condition": "Initial (t=0)",
        "Value": f"{C_at_t_start.mean().item():.6f}",
        "Expected": "0",
        "Difference":f"{(C_at_t_start- 0.0).abs().mean().item():.6f}"
    }
]

# 创建并显示表格
df = pd.DataFrame(boundary_data)
print("\nBoundary Conditions Verification:")
print(df.to_string(index=False))

# 保存为CSV
df.to_csv(os.path.join(SAVE_DIR, "boundary_conditions.csv"), index=False)

# ================= 保存所有结果 =================
print(f"\nAll analysis results saved to: {SAVE_DIR}")
print(f"Total execution time: {time.time()-start_time:.2f} seconds")